## Dataset Features

### **Target Variable**

### `SeriousDlqin2yrs`
Indicates whether a customer became **90 or more days past due** on at least one payment within the next **2 years**.

- **0** → No default
- **1** → Default

This is the **target (label)** that our model will predict.

---

### **Input Features**

### `RevolvingUtilizationOfUnsecuredLines`
The proportion of available **unsecured credit** (e.g., credit cards and personal lines of credit) currently being used.

**Formula:**

\[
\text{Credit Utilization}=\frac{\text{Current Balance}}{\text{Credit Limit}}
\]

- Higher values generally indicate greater financial stress.
- Expected relationship: **Higher utilization → Higher default risk.**

---

### `age`
The customer's age in years.

- Younger borrowers may have shorter credit histories.
- Older borrowers often have more stable financial profiles.

Expected relationship: **Age may have a non-linear relationship with default risk.**

---

### `NumberOfTime30-59DaysPastDueNotWorse`
Number of times the customer was **30–59 days late** on a payment.

- Indicates moderate payment delinquency.
- Expected relationship: **More late payments → Higher default risk.**

---

### `DebtRatio`
Measures the customer's debt burden relative to income.

**Simplified formula:**

\[
\text{Debt Ratio}=\frac{\text{Monthly Debt Payments}}{\text{Monthly Income}}
\]

- Higher values indicate a larger portion of income is committed to debt.
- Expected relationship: **Higher debt ratio → Higher default risk.**

> **Note:** In this dataset, values greater than 1 may occur due to the dataset's calculation method and data quality. These will be investigated during EDA.

---

### `MonthlyIncome`
Customer's monthly income.

- Represents repayment capacity.
- Expected relationship: **Higher income generally reduces default risk.**

---

### `NumberOfOpenCreditLinesAndLoans`
Total number of open credit accounts, such as credit cards and loans.

- Reflects the customer's current credit exposure.
- Very high or very low values may indicate increased risk.

---

### `NumberOfTimes90DaysLate`
Number of times the customer was **90 or more days late** on payments.

- Represents severe payment delinquency.
- Expected relationship: **One of the strongest predictors of future default.**

---

### `NumberRealEstateLoansOrLines`
Number of real estate loans or mortgage credit lines.

- Indicates property-related borrowing.
- May reflect either greater debt or stronger financial stability, depending on the borrower.

---

### `NumberOfTime60-89DaysPastDueNotWorse`
Number of times the customer was **60–89 days late** on payments.

- Represents significant payment delinquency.
- Expected relationship: **Higher values increase default risk.**

---

### `NumberOfDependents`
Number of people financially dependent on the customer.

- Represents financial responsibility.
- May increase financial pressure, especially when income is limited.
- Can later be combined with income to create engineered features such as **Income per Dependent**.

### Dataset Overview

In [ ]:
import pandas as pd
# Load dataset
df = pd.read_csv("../data/raw/cs-training.csv", index_col=0)

# Display the first five rows
df.head()

In [ ]:
# Number of rows and columns
rows, cols = df.shape

print(f"Rows    : {rows:,}")
print(f"Columns : {cols}")

### Observations

- The dataset contains **150,000 customer records**, providing sufficient data for robust model training and validation.
- There are **11 columns**, consisting of **10 input features** and **1 binary target variable (`SeriousDlqin2yrs`)**.
- The dataset is relatively low-dimensional, making it well-suited for both linear models and tree-based ensemble methods.

In [ ]:
# Display all column names
print("Columns in the dataset:\n")

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

In [ ]:
# Target column
TARGET = "SeriousDlqin2yrs"

print(f"Target Column : {TARGET}")
print(f"Data Type     : {df[TARGET].dtype}")
print(f"Unique Values : {sorted(df[TARGET].unique())}")

In [ ]:
# Dataset information
df.info()

### Observations

- All features have appropriate numeric data types.
- No object (string) columns are present.
- `MonthlyIncome` and `NumberOfDependents` contain missing values.
- The remaining features are complete.
- The dataset occupies a relatively small amount of memory and can be processed efficiently in memory.

In [ ]:
# Count duplicate rows

duplicate_rows = df.duplicated().sum()

print(f"Duplicate Rows: {duplicate_rows}")
print(f"Duplicate Percentage: {duplicate_rows / len(df) * 100:.2f}%")

In [ ]:
# Dataset quality summary

summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Non-Null": df.count(),
    "Missing": df.isnull().sum(),
    "Missing (%)": (df.isnull().sum() / len(df) * 100).round(2),
    "Unique Values": df.nunique()
})

summary


### Dataset Overview

- The dataset contains **150,000 records** and **11 numerical features** (10 predictors and 1 target).
- All features are numeric (`int64` or `float64`); no categorical encoding is required.
- `MonthlyIncome` (**19.82%**) and `NumberOfDependents` (**2.62%**) contain missing values and will require further investigation.
- No constant features were found, and all variables contain useful variation.
- The dataset uses only **13.7 MB** of memory, making it efficient to process in memory.

### Duplicate Records

- Approximately **0.41%** of the rows are duplicates.
- Since the dataset does not include a unique customer identifier, it is not possible to determine whether these are true duplicate customers or different customers with identical financial profiles.
- Duplicate rows will be **retained for now** and revisited only if later analysis provides evidence that they are data-entry errors.

In [ ]:
# Target class distribution

target_counts = df["SeriousDlqin2yrs"].value_counts().sort_index()

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": (target_counts / len(df) * 100).round(2)
})

target_summary.index = ["No Default (0)", "Default (1)"]

target_summary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))

ax = df["SeriousDlqin2yrs"].value_counts().sort_index().plot(
    kind="bar",
    width=0.6
)

plt.title("Target Class Distribution")
plt.xlabel("Target Class")
plt.ylabel("Number of Customers")
plt.xticks([0, 1], ["No Default (0)", "Default (1)"])

# Add value labels
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom"
    )

plt.show()

In [ ]:


# Target distribution
target_counts = df["SeriousDlqin2yrs"].value_counts().sort_index()

labels = ["No Default (0)", "Default (1)"]

plt.figure(figsize=(6, 6))

plt.pie(
    target_counts,
    labels=labels,
    autopct="%1.2f%%",
    startangle=90,
)

plt.title("Target Class Distribution")

plt.axis("equal")  # Makes the pie circular

plt.show()

### Observations

- The dataset is **highly imbalanced**.
- Approximately **93.32%** of customers did not default, while only **6.68%** experienced serious delinquency.
- This imbalance means that a model can achieve high accuracy by predicting the majority class only, making accuracy an unreliable evaluation metric.
- Special attention must be given to identifying the minority class (defaults) during model development.

### Metric Selection

- **Accuracy is not an appropriate primary metric** because the dataset is highly imbalanced.
- The project will prioritize **Recall** to maximize the detection of defaulting customers.
- To avoid excessive false positives, Recall will be evaluated alongside a **minimum Precision threshold of 0.70**.
- **ROC-AUC** and **PR-AUC** will be used to assess the model's ranking performance across different decision thresholds.
- **F1-score** will be reported as a secondary metric, while Accuracy will be treated as supplementary information only.

### Feature Audit (One Feature at a Time)

## Feature: RevolvingUtilizationOfUnsecuredLines

### Business Understanding

**Definition**

`RevolvingUtilizationOfUnsecuredLines` measures the proportion of a customer's available unsecured revolving credit that is currently being used.

\[
\text{Utilization}=\frac{\text{Outstanding Balance}}{\text{Total Credit Limit}}
\]

For example, if a customer has a credit limit of \$10,000 and currently owes \$3,000, the utilization is **0.30 (30%)**.

### Why should it influence default?

Customers with high credit utilization are often under greater financial stress and may have a higher likelihood of missing future payments.

### Expected Relationship

We expect a **positive relationship** between utilization and default risk: as utilization increases, the probability of default should generally increase. Extremely high utilization values will be investigated during EDA to determine whether they represent legitimate financial behavior or data anomalies.

In [ ]:
# Quality check: RevolvingUtilizationOfUnsecuredLines
feature = "RevolvingUtilizationOfUnsecuredLines"

quality_summary = pd.DataFrame({
    "Data Type": [df[feature].dtype],
    "Missing Values": [df[feature].isna().sum()],
    "Missing (%)": [round(df[feature].isna().mean() * 100, 2)],
    "Negative Values": [(df[feature] < 0).sum()],
    "Minimum": [df[feature].min()],
    "Maximum": [df[feature].max()],
    "Unique Values": [df[feature].nunique()]
}, index=[feature])

quality_summary

In [ ]:
# Check how many utilization values exceed 100%

(df[feature] > 1).sum()

In [ ]:
# Display the 10 largest utilization values

df[feature].sort_values(ascending=False).head(10)

### Quality Check Observations

- `RevolvingUtilizationOfUnsecuredLines` is stored as a continuous (`float64`) variable.
- The feature contains **no missing values** and **no negative values**.
- Values range from **0.0** to **50,708.0**.
- **3,321 observations (2.21%)** have utilization values greater than **1.0 (100%)**, indicating unusually high utilization.
- These extreme values are not removed at this stage. They will be investigated further during distribution analysis, outlier analysis, and preprocessing before any transformation or clipping decisions are made.

In [ ]:
# Statistical summary: RevolvingUtilizationOfUnsecuredLines

feature = "RevolvingUtilizationOfUnsecuredLines"

summary = pd.DataFrame({
    "Mean": [df[feature].mean()],
    "Median": [df[feature].median()],
    "Mode": [df[feature].mode()[0]],
    "Variance": [df[feature].var()],
    "Standard Deviation": [df[feature].std()],
    "Minimum": [df[feature].min()],
    "25%": [df[feature].quantile(0.25)],
    "50%": [df[feature].quantile(0.50)],
    "75%": [df[feature].quantile(0.75)],
    "Maximum": [df[feature].max()],
    "Skewness": [df[feature].skew()],
    "Kurtosis": [df[feature].kurt()]
}, index=[feature])

summary.T

### Statistical Analysis Observations

- The feature exhibits **extreme positive skewness** (97.63), indicating a long right tail driven by a small number of very large utilization values.
- The **mean (6.05)** is substantially higher than the **median (0.154)**, confirming that extreme observations heavily influence the average.
- The **mode is 0.0**, suggesting many customers have no revolving balance.
- Quartiles show that **75% of customers have utilization below 0.56**, while the maximum value reaches **50,708**, indicating the presence of extreme outliers.
- The very high **standard deviation (249.76)** and **kurtosis (14,544.71)** further confirm that the distribution is highly dispersed with exceptionally heavy tails.
- No preprocessing decisions are made at this stage; the feature will be examined visually before considering transformations or outlier treatment.

In [ ]:
# Histogram: RevolvingUtilizationOfUnsecuredLines


feature = "RevolvingUtilizationOfUnsecuredLines"

plt.figure(figsize=(8, 5))

plt.hist(df[feature], bins=50)

plt.title(f"Histogram of {feature}")
plt.xlabel(feature)
plt.ylabel("Frequency")

plt.show()

In [ ]:
# Kernel Density Estimate (KDE): RevolvingUtilizationOfUnsecuredLines



feature = "RevolvingUtilizationOfUnsecuredLines"

plt.figure(figsize=(8, 5))

df[feature].plot(kind="density")

plt.title(f"KDE of {feature}")
plt.xlabel(feature)

plt.show()

In [ ]:
# Boxplot: RevolvingUtilizationOfUnsecuredLines


feature = "RevolvingUtilizationOfUnsecuredLines"

plt.figure(figsize=(10, 2.5))

plt.boxplot(df[feature], vert=False)

plt.title(f"Boxplot of {feature}")
plt.xlabel(feature)

plt.show()


In [ ]:
# Violin plot: RevolvingUtilizationOfUnsecuredLines



feature = "RevolvingUtilizationOfUnsecuredLines"

plt.figure(figsize=(8, 3))

plt.violinplot(df[feature], vert=False)

plt.title(f"Violin Plot of {feature}")
plt.xlabel(feature)

plt.show()

In [ ]:
# Zoomed histogram: RevolvingUtilizationOfUnsecuredLines (0–99th percentile)



feature = "RevolvingUtilizationOfUnsecuredLines"

upper = df[feature].quantile(0.99)

plt.figure(figsize=(8, 5))

plt.hist(
    df.loc[df[feature] <= upper, feature],
    bins=50
)

plt.title(f"{feature} (0–99th Percentile)")
plt.xlabel(feature)
plt.ylabel("Frequency")

plt.show()

In [ ]:
# Zoomed KDE: RevolvingUtilizationOfUnsecuredLines (0–99th percentile)


feature = "RevolvingUtilizationOfUnsecuredLines"

upper = df[feature].quantile(0.99)

plt.figure(figsize=(8, 5))

df.loc[df[feature] <= upper, feature].plot(kind="density")

plt.title(f"{feature} (0–99th Percentile)")
plt.xlabel(feature)

plt.show()

In [ ]:
# Boxplot without extreme axis stretching


feature = "RevolvingUtilizationOfUnsecuredLines"

upper = df[feature].quantile(0.99)

plt.figure(figsize=(10, 2.5))

plt.boxplot(df.loc[df[feature] <= upper, feature], vert=False)

plt.title(f"{feature} (0–99th Percentile)")
plt.xlabel(feature)

plt.show()

### Distribution Analysis Observations

- The distribution is **highly right-skewed**, with most observations concentrated near zero utilization.
- A large proportion of customers have little or no revolving credit utilization, resulting in a pronounced peak near **0**.
- A second noticeable concentration appears around **1.0 (100% utilization)**, suggesting that many customers are using nearly all of their available revolving credit.
- The feature exhibits a long right tail, consistent with the large skewness and kurtosis observed earlier.
- Extreme values beyond the 99th percentile compress the original visualization; therefore, a zoomed plot provides a more representative view of the majority of observations.
- The apparent second peak near **1.0** will be investigated further using target analysis and potential feature engineering.

In [ ]:
# Detect outliers using the IQR method

feature = "RevolvingUtilizationOfUnsecuredLines"

Q1 = df[feature].quantile(0.25)
Q3 = df[feature].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df[feature] < lower_bound) |
    (df[feature] > upper_bound)
]

print(f"Q1           : {Q1:.4f}")
print(f"Q3           : {Q3:.4f}")
print(f"IQR          : {IQR:.4f}")
print(f"Lower Bound  : {lower_bound:.4f}")
print(f"Upper Bound  : {upper_bound:.4f}")
print(f"Outliers     : {len(outliers):,}")
print(f"Percentage   : {len(outliers)/len(df)*100:.2f}%")

In [ ]:
# Display the largest outlier values

outliers.sort_values(feature, ascending=False).head(10)

In [ ]:
# Summary statistics for outliers

outliers[feature].describe()

In [ ]:
# Compare default rate for outliers vs non-outliers

feature = "RevolvingUtilizationOfUnsecuredLines"

Q1 = df[feature].quantile(0.25)
Q3 = df[feature].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR

df["is_outlier"] = (
    (df[feature] < lower_bound) |
    (df[feature] > upper_bound)
)

comparison = (
    df.groupby("is_outlier")["SeriousDlqin2yrs"]
      .agg(
          Count="count",
          Defaults="sum",
          Default_Rate="mean"
      )
)

comparison["Default_Rate"] *= 100

comparison

### Outlier Analysis Observations

- Using the IQR method, **763 observations (0.51%)** were identified as outliers.
- Although these observations are statistically extreme, they are **highly informative** from a business perspective.
- Customers classified as outliers have a **default rate of 30.80%**, compared with **6.56%** for non-outliers—approximately **4.7 times higher**.
- This indicates that extreme revolving credit utilization is strongly associated with credit default and should **not** be removed during EDA.
- The feature will be retained. Possible transformations or clipping strategies will be evaluated later during preprocessing based on validation performance rather than applied immediately.

In [ ]:
# Compare feature statistics by target class

feature = "RevolvingUtilizationOfUnsecuredLines"
target = "SeriousDlqin2yrs"

df.groupby(target)[feature].describe().round(3)

In [ ]:
# Boxplot of feature grouped by target

import matplotlib.pyplot as plt

feature = "RevolvingUtilizationOfUnsecuredLines"
target = "SeriousDlqin2yrs"

plt.figure(figsize=(8,5))

df.boxplot(column=feature, by=target)

plt.title(f"{feature} by {target}")
plt.suptitle("")
plt.xlabel("Default")
plt.ylabel(feature)

plt.show()

In [ ]:
# Zoomed boxplot (0–99th percentile)

feature = "RevolvingUtilizationOfUnsecuredLines"
target = "SeriousDlqin2yrs"

upper = df[feature].quantile(0.99)

plt.figure(figsize=(8,5))

df[df[feature] <= upper].boxplot(
    column=feature,
    by=target
)

plt.title(f"{feature} by {target} (0–99th Percentile)")
plt.suptitle("")
plt.xlabel("Default")
plt.ylabel(feature)

plt.show()

### Relationship with Target Observations

- The distribution of `RevolvingUtilizationOfUnsecuredLines` differs substantially between defaulters and non-defaulters.
- Although the mean utilization is higher for non-defaulters, this result is driven by a small number of extremely large values and is therefore not representative.
- The **median utilization** for defaulters (0.839) is approximately **6.3 times higher** than for non-defaulters (0.133).
- Quartiles show that the entire utilization distribution is shifted toward higher values for customers who defaulted.
- These findings suggest that higher revolving credit utilization is strongly associated with increased default risk, making this feature an important candidate for the predictive model.

In [ ]:
# Create business-friendly utilization bins

feature = "RevolvingUtilizationOfUnsecuredLines"
target = "SeriousDlqin2yrs"

bins = [-0.01, 0.10, 0.30, 0.50, 0.80, 1.00, float("inf")]

labels = [
    "0–10%",
    "10–30%",
    "30–50%",
    "50–80%",
    "80–100%",
    ">100%"
]

df["Utilization_Bin"] = pd.cut(
    df[feature],
    bins=bins,
    labels=labels
)


In [ ]:
# Calculate default rate for each utilization bin

bin_summary = (
    df.groupby("Utilization_Bin", observed=True)[target]
      .agg(
          Customers="count",
          Defaults="sum",
          Default_Rate="mean"
      )
)

bin_summary["Default_Rate"] *= 100
bin_summary = bin_summary.round(2)

bin_summary

In [ ]:
# Plot default rate by utilization bin


plt.figure(figsize=(8,5))

plt.bar(
    bin_summary.index.astype(str),
    bin_summary["Default_Rate"]
)

plt.title("Default Rate by Credit Utilization")
plt.xlabel("Credit Utilization")
plt.ylabel("Default Rate (%)")

plt.show()

In [ ]:
# Number of customers in each utilization bin

plt.figure(figsize=(8,5))

plt.bar(
    bin_summary.index.astype(str),
    bin_summary["Customers"]
)

plt.title("Customers by Credit Utilization")
plt.xlabel("Credit Utilization")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    bin_summary.index.astype(str),
    bin_summary["Default_Rate"],
    marker="o",
    linewidth=2
)

plt.title("Default Rate vs Credit Utilization")
plt.xlabel("Credit Utilization")
plt.ylabel("Default Rate (%)")

plt.grid(alpha=0.3)

plt.show()

In [ ]:

x = [0.05, 0.20, 0.40, 0.65, 0.90, 1.20]
y = bin_summary["Default_Rate"].values

plt.figure(figsize=(8,5))

plt.plot(x, y, marker="o", linewidth=2)

plt.xticks(
    x,
    ["0-10%", "10-30%", "30-50%", "50-80%", "80-100%", ">100%"]
)

plt.title("Default Rate vs Credit Utilization")
plt.xlabel("Credit Utilization")
plt.ylabel("Default Rate (%)")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# Compare observed trend with a linear fit

import numpy as np

x = np.array([0.05, 0.20, 0.40, 0.65, 0.90, 1.20])
y = bin_summary["Default_Rate"].values

coef = np.polyfit(x, y, 1)
linear_fit = np.polyval(coef, x)

plt.figure(figsize=(8,5))

plt.plot(x, y, "o-", label="Observed")
plt.plot(x, linear_fit, "--", label="Linear Fit")

plt.xticks(
    x,
    ["0-10%", "10-30%", "30-50%", "50-80%", "80-100%", ">100%"]
)

plt.xlabel("Credit Utilization")
plt.ylabel("Default Rate (%)")
plt.title("Observed Trend vs Linear Fit")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

### Binning Analysis Observations

- The default rate increases consistently as revolving credit utilization increases, indicating a **strong monotonic relationship** between utilization and credit default.
- Customers with **0–10% utilization** have a default rate of only **1.81%**, while customers with **utilization greater than 100%** have a default rate of **37.25%**.
- Borrowers exceeding 100% utilization are approximately **20 times more likely** to default than borrowers using less than 10% of their available credit.
- The increase in default risk accelerates at higher utilization levels, suggesting a **nonlinear relationship**.
- This feature is a strong predictor of credit default and provides several opportunities for future feature engineering, including high-utilization flags, over-limit indicators, and utilization categories.

In [ ]:
# Pearson correlation with target

feature = "RevolvingUtilizationOfUnsecuredLines"
target = "SeriousDlqin2yrs"

pearson_corr = df[[feature, target]].corr(method="pearson").iloc[0, 1]

print(f"Pearson Correlation: {pearson_corr:.4f}")

In [ ]:
# Spearman correlation with target

feature = "RevolvingUtilizationOfUnsecuredLines"
target = "SeriousDlqin2yrs"

spearman_corr = df[[feature, target]].corr(method="spearman").iloc[0, 1]

print(f"Spearman Correlation: {spearman_corr:.4f}")

### Correlation Analysis

- Pearson correlation between `RevolvingUtilizationOfUnsecuredLines` and the target is **-0.0018**, indicating almost no linear relationship.
- This result is expected because the feature contains extremely large outliers that heavily influence Pearson correlation.
- Spearman correlation is **0.2404**, indicating a weak-to-moderate positive monotonic relationship.
- The Spearman result aligns with the binning analysis, where default rates consistently increase as utilization increases.
- For this feature, Spearman provides a more meaningful measure of association than Pearson because it is robust to outliers and captures monotonic trends.

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐⭐⭐ Very High |
| Predictive Signal | ⭐⭐⭐⭐⭐ Very Strong |
| Missing Values | None |
| Invalid Values | None |
| Negative Values | None |
| Distribution | Extremely Right-Skewed |
| Outliers | Present but Informative |
| Correlation | Spearman (0.2404) is more meaningful than Pearson (-0.0018) due to extreme outliers. |
| Feature Engineering | High Utilization Flag, Over-Limit Flag, Utilization Categories |
| Preprocessing Recommendation | Evaluate clipping and Yeo-Johnson/Log transformation. Do not remove outliers without validation evidence. |

### Final Verdict

`RevolvingUtilizationOfUnsecuredLines` is one of the strongest predictors in the dataset.

Although the feature contains extreme outliers and exhibits severe right skewness, these observations correspond to customers with substantially higher default rates and therefore contain valuable business information rather than noise.

The feature will be **retained** for model development. Potential transformations and engineered features will be evaluated during the preprocessing and feature engineering phases based on validation performance.

# re-useable code snippets

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

In [ ]:
# ===============================
# Feature Quality Report
# ===============================

def quality_report(df, feature):
    report = pd.DataFrame({
        "Data Type": [df[feature].dtype],
        "Missing Values": [df[feature].isna().sum()],
        "Missing (%)": [round(df[feature].isna().mean()*100,2)],
        "Negative Values": [(df[feature] < 0).sum()],
        "Minimum": [df[feature].min()],
        "Maximum": [df[feature].max()],
        "Unique Values": [df[feature].nunique()]
    }, index=[feature])

    display(report)

In [ ]:
# ===============================
# Statistical Summary
# ===============================

def statistical_summary(df, feature):

    data = df[feature].dropna()

    summary = pd.DataFrame({
        feature:[
            data.mean(),
            data.median(),
            data.mode()[0],
            data.var(),
            data.std(),
            data.min(),
            data.quantile(.25),
            data.quantile(.50),
            data.quantile(.75),
            data.max(),
            stats.skew(data),
            stats.kurtosis(data)
        ]
    }, index=[
        "Mean",
        "Median",
        "Mode",
        "Variance",
        "Standard Deviation",
        "Minimum",
        "25%",
        "50%",
        "75%",
        "Maximum",
        "Skewness",
        "Kurtosis"
    ])

    display(summary)

In [ ]:
# ===============================
# Distribution Plots
# ===============================

def distribution_plots(df, feature):

    fig, ax = plt.subplots(2,2, figsize=(12,8))

    sns.histplot(df[feature], kde=True, ax=ax[0,0])

    sns.boxplot(x=df[feature], ax=ax[0,1])

    sns.violinplot(x=df[feature], ax=ax[1,0])

    sns.kdeplot(df[feature], fill=True, ax=ax[1,1])

    ax[0,0].set_title("Histogram")
    ax[0,1].set_title("Boxplot")
    ax[1,0].set_title("Violin")
    ax[1,1].set_title("KDE")

    plt.tight_layout()
    plt.show()

In [ ]:
# ===============================
# Outlier Analysis
# ===============================

def outlier_analysis(df, feature, target):

    temp = df.copy()

    Q1 = temp[feature].quantile(.25)
    Q3 = temp[feature].quantile(.75)

    IQR = Q3-Q1

    lower = Q1-1.5*IQR
    upper = Q3+1.5*IQR

    temp["is_outlier"] = (
        (temp[feature]<lower) |
        (temp[feature]>upper)
    )

    summary = temp.groupby("is_outlier")[target].agg(
        Customers="count",
        Defaults="sum",
        Default_Rate="mean"
    )

    summary["Default_Rate"]*=100

    display(summary.round(2))

In [ ]:
# ===============================
# Relationship With Target
# ===============================

def target_relationship(df, feature, target):

    display(
        df.groupby(target)[feature].describe().round(3)
    )

    plt.figure(figsize=(7,4))

    sns.boxplot(
        data=df,
        x=target,
        y=feature
    )

    plt.show()

In [ ]:
# ===============================
# Binning Analysis
# ===============================

def binning_analysis(df, feature, target, bins=6):

    temp=df.copy()

    temp["Bin"]=pd.qcut(
        temp[feature],
        q=bins,
        duplicates="drop"
    )

    summary=temp.groupby("Bin")[target].agg(
        Customers="count",
        Defaults="sum",
        Default_Rate="mean"
    )

    summary["Default_Rate"]*=100

    display(summary.round(2))

    plt.figure(figsize=(8,4))

    plt.plot(
        summary.index.astype(str),
        summary["Default_Rate"],
        marker="o"
    )

    plt.xticks(rotation=30)

    plt.ylabel("Default Rate (%)")

    plt.show()

In [ ]:
# ===============================
# Correlation Analysis
# ===============================

def correlation_analysis(df, feature, target):

    pearson=df[[feature,target]].corr(
        method="pearson"
    ).iloc[0,1]

    spearman=df[[feature,target]].corr(
        method="spearman"
    ).iloc[0,1]

    corr=pd.DataFrame({
        "Correlation":[pearson,spearman]
    }, index=[
        "Pearson",
        "Spearman"
    ])

    display(corr.round(4))

In [ ]:
# ===============================
# Complete Feature Audit
# ===============================

def feature_audit(df, feature, target):

    print("="*80)
    print(feature.upper())
    print("="*80)

    print("\n1. Quality Report")
    quality_report(df, feature)

    print("\n2. Statistical Summary")
    statistical_summary(df, feature)

    print("\n3. Distribution")
    distribution_plots(df, feature)

    print("\n4. Outlier Analysis")
    outlier_analysis(df, feature, target)

    print("\n5. Relationship With Target")
    target_relationship(df, feature, target)

    print("\n6. Binning Analysis")
    binning_analysis(df, feature, target)

    print("\n7. Correlation Analysis")
    correlation_analysis(df, feature, target)

In [ ]:
feature_audit(
    df,
    feature="age",
    target="SeriousDlqin2yrs"
)

# Feature Summary: age

## Business Understanding

- `age` represents the customer's age in years at the time of credit evaluation.
- Age is an important demographic feature that often reflects **financial stability, employment experience, income consistency, and length of credit history**.
- Younger borrowers generally have shorter credit histories and less financial stability, while older borrowers often have more established financial behavior.
- **Expected Relationship:** As age increases, the probability of default is generally expected to decrease, although the relationship may not be perfectly linear.

---

## Data Quality

- Missing Values: **0 (0.00%)**
- Negative Values: **0**
- Data Type: **int64**
- Unique Values: **86**
- Value Range: **0 – 109 years**

### Observations

- The feature is complete with **no missing values** and **no negative values**, indicating excellent overall data quality.
- One observation has an age of **0 years**, which is not a valid age for a credit applicant and will be investigated during preprocessing.
- The maximum age of **109 years** is uncommon but still plausible and will be retained unless further evidence suggests otherwise.

---

## Statistical Analysis

- Mean: **52.30**
- Median: **52**
- Mode: **49**
- Standard Deviation: **14.77**
- Skewness: **0.19**
- Kurtosis: **-0.49**

### Observations

- Mean and median are nearly identical, indicating a well-centered distribution.
- The feature exhibits **very slight positive skewness**, suggesting the distribution is approximately symmetric.
- Negative kurtosis indicates a slightly flatter distribution than a normal distribution.
- Unlike highly skewed variables such as credit utilization, **age does not require transformation**.

---

## Distribution Analysis

### Observations

- The age distribution is approximately bell-shaped with no extreme skewness.
- Most customers are concentrated between **40 and 65 years of age**.
- Very young and very old customers represent only a small portion of the dataset.
- The distribution is suitable for both linear and tree-based machine learning models without additional transformation.

---

## Outlier Analysis

- Outliers Identified: **46 (0.03%)**
- Default Rate (Non-Outliers): **6.68%**
- Default Rate (Outliers): **6.52%**

### Observations

- Only a very small number of observations were identified as outliers using the IQR method.
- Outlier customers have nearly the same default rate as the remaining population.
- These observations do not appear to represent unusually risky customers.

**Decision:** Retain all observations. No outlier treatment is required.

---

## Relationship with Target

- Median Age (No Default): **52 years**
- Median Age (Default): **45 years**

### Observations

- Customers who default are generally younger than customers who successfully repay their loans.
- The entire age distribution for defaulters is shifted toward younger ages.
- This suggests that younger borrowers carry higher credit risk within this dataset.

---

## Binning Analysis

| Age Group | Default Rate |
|-----------|-------------:|
| ≤37 | 10.82% |
| 37–45 | 8.68% |
| 45–52 | 7.90% |
| 52–59 | 5.96% |
| 59–67 | 3.92% |
| >67 | 2.25% |

### Observations

- Default risk decreases consistently as age increases.
- The youngest borrowers have nearly **five times** the default rate of the oldest borrowers.
- The trend is smooth and monotonic, indicating that age provides meaningful predictive information.
- This pattern also suggests opportunities for future feature engineering using age categories or risk groups.

---

## Correlation Analysis

- Pearson Correlation: **-0.1154**
- Spearman Correlation: **-0.1170**

### Observations

- Both correlation coefficients indicate a **weak negative relationship** between age and default.
- Pearson and Spearman correlations are nearly identical, suggesting that outliers have little influence on this feature.
- The negative correlation supports the business observation that **older customers are less likely to default**.

---

## Feature Engineering Opportunities

Potential engineered features include:

- Young Borrower Flag
- Age Categories
- Risk-Based Age Groups
- Nonlinear Age Bins

These engineered features will only be retained if they improve cross-validation performance.

---

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐⭐☆ High |
| Predictive Signal | ⭐⭐⭐⭐☆ Moderate–High |
| Missing Values | None |
| Invalid Values | One record (Age = 0) |
| Distribution | Approximately Normal |
| Outliers | Present but Not Informative |
| Correlation | Weak Negative (Pearson ≈ Spearman) |
| Feature Engineering | Age Groups, Young Borrower Flag |
| Preprocessing Recommendation | Investigate invalid age (0); otherwise no transformation or outlier treatment required. |

### Final Verdict

`age` is a **clean, interpretable, and reliable predictor** of credit default. It requires minimal preprocessing, exhibits a stable monotonic relationship with the target, and aligns well with domain knowledge that younger borrowers generally present higher credit risk. The feature will be retained in its original form, while age-based engineered features may be explored during the feature engineering phase.

In [ ]:
feature_audit(
    df,
    feature="NumberOfTime30-59DaysPastDueNotWorse",
    target="SeriousDlqin2yrs"
)

In [ ]:
bins = [-1, 0, 1, 2, 5, 10, np.inf]
labels = ["0", "1", "2", "3-5", "6-10", ">10"]

df["Late30_Bin"] = pd.cut(
    df["NumberOfTime30-59DaysPastDueNotWorse"],
    bins=bins,
    labels=labels
)

bin_summary = (
    df.groupby("Late30_Bin", observed=True)["SeriousDlqin2yrs"]
      .agg(Customers="count", Defaults="sum")
)

bin_summary["Default_Rate"] = (
    bin_summary["Defaults"] / bin_summary["Customers"] * 100
).round(2)

display(bin_summary)

In [ ]:
plt.figure(figsize=(7,4))

sns.barplot(
    x=bin_summary.index,
    y=bin_summary["Default_Rate"]
)

plt.title("Default Rate by 30–59 Day Delinquency Count")
plt.xlabel("Late Payment Count")
plt.ylabel("Default Rate (%)")
plt.show()

# Feature Summary: NumberOfTime30-59DaysPastDueNotWorse

## Business Understanding

- Represents the number of times a customer was **30–59 days past due** on a payment without progressing to a more severe delinquency.
- This feature measures a customer's recent repayment behavior and is one of the strongest indicators of future credit risk.
- Customers with repeated late payments demonstrate poor payment discipline and are generally more likely to default.
- **Expected Relationship:** As the number of late payments increases, the probability of default is expected to increase.

---

## Data Quality

- Missing Values: **0 (0.00%)**
- Negative Values: **0**
- Data Type: **int64**
- Unique Values: **16**
- Value Range: **0 – 98**

### Observations

- The feature contains no missing or invalid values.
- Most customers have zero late payments, while only a small proportion have multiple delinquencies.
- Extremely large values (e.g., 96 or 98) are known to occur in this dataset and will be investigated during preprocessing rather than removed during EDA.

---

## Statistical Analysis

- Mean: **0.42**
- Median: **0**
- Mode: **0**
- Standard Deviation: **4.19**
- Skewness: **22.60**
- Kurtosis: **522.36**

### Observations

- The feature is **extremely right-skewed**, with the majority of customers having no late payments.
- The large difference between the mean and median indicates that only a small number of customers have multiple delinquencies.
- Extremely high kurtosis reflects the presence of a long right tail and several extreme observations.
- The distribution is highly non-normal and may benefit from feature engineering rather than direct transformation.

---

## Distribution Analysis

### Observations

- The distribution contains a very large spike at **0**, indicating that most customers have never been 30–59 days late.
- A long right tail is formed by customers with repeated delinquency.
- This distribution is expected for repayment history variables and represents genuine customer behavior rather than poor data quality.

---

## Outlier Analysis

- Default Rate (Non-Outliers): **4.00%**
- Default Rate (Outliers): **20.79%**

### Observations

- Customers identified as outliers have a default rate more than **five times higher** than the remaining population.
- These observations represent genuinely high-risk borrowers rather than data errors.

**Decision:** Keep all outliers. They contain valuable predictive information.

---

## Relationship with Target

- Mean Late Payments (No Default): **0.28**
- Mean Late Payments (Default): **2.39**

### Observations

- Customers who default have substantially more previous 30–59 day late payments.
- On average, defaulters have approximately **8.5 times** more late payments than non-defaulters.
- This demonstrates that repayment history is a strong discriminator between low-risk and high-risk customers.

---

## Binning Analysis

| Late Payments | Default Rate |
|--------------|-------------:|
| 0 | **4.00%** |
| 1 | **15.03%** |
| 2 | **26.51%** |
| 3–5 | **38.34%** |
| 6–10 | **49.79%** |
| >10 | **54.95%** |

### Observations

- A clear monotonic relationship exists between late-payment frequency and default risk.
- Customers with **no previous late payments** have only a **4.00%** default rate.
- Even **one late payment** nearly quadruples the default risk.
- Default probability continues to increase steadily as delinquency frequency rises, reaching **54.95%** for customers with more than ten late payments.
- This strong risk gradient confirms that previous delinquency is one of the most powerful behavioral predictors in the dataset.
- Business-defined bins provided a much more meaningful interpretation than automatic quantile binning because the feature is highly concentrated at zero.

---

## Correlation Analysis

- Pearson Correlation: **0.1256**
- Spearman Correlation: **0.2574**

### Observations

- Both correlation measures indicate a positive relationship with default.
- Spearman correlation is noticeably stronger because it captures the monotonic increase in default risk across increasing delinquency counts.
- The difference between Pearson and Spearman reflects the feature's highly skewed, non-linear distribution.

---

## Feature Engineering Opportunities

Potential engineered features include:

- Any Late Payment Flag
- Frequent Late Payment Flag
- High Delinquency Flag
- Delinquency Categories
- Total Delinquency Count (combined with other late-payment variables)
- Maximum Delinquency Indicator

---

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐⭐⭐ Very High |
| Predictive Signal | ⭐⭐⭐⭐⭐ Very Strong |
| Missing Values | None |
| Invalid Values | None |
| Distribution | Extremely Right-Skewed |
| Outliers | Highly Informative (Keep) |
| Correlation | Moderate Positive (Spearman > Pearson) |
| Binning | Strong Monotonic Risk Trend |
| Feature Engineering | Very High Priority |
| Preprocessing Recommendation | Retain outliers, investigate extreme coded values, and use business-defined delinquency bins for feature engineering. |

### Final Verdict

`NumberOfTime30-59DaysPastDueNotWorse` is one of the strongest behavioral predictors of credit default in the dataset. It exhibits a clear monotonic relationship with default risk, where increasing delinquency frequency consistently corresponds to higher default rates. The feature contains highly informative outliers, aligns closely with credit-risk domain knowledge, and offers excellent opportunities for feature engineering. It will be retained without removing outliers and will play a significant role in the final predictive model.

In [ ]:
# ==========================================================
# Feature Audit: DebtRatio
# ==========================================================

feature_audit(
    df=df,
    feature="DebtRatio",
    target="SeriousDlqin2yrs"
)

# Feature Summary: DebtRatio

## Business Understanding

- `DebtRatio` represents the proportion of a customer's monthly debt obligations relative to their monthly income.
- It measures the customer's financial burden and repayment capacity.
- A higher debt ratio generally indicates greater financial stress and a potentially higher risk of default.
- **Expected Relationship:** As DebtRatio increases, the probability of default is expected to increase.

---

## Data Quality

- Missing Values: **0 (0.00%)**
- Negative Values: **0**
- Data Type: **float64**
- Unique Values: **114,194**
- Value Range: **0 – 329,664**

### Observations

- The feature contains no missing or negative values.
- Extremely large values are present, producing a very wide range.
- These extreme values are likely caused by customers with very low reported income or other financial reporting characteristics rather than data entry errors.
- Further investigation of these values will be performed during preprocessing.

---

## Statistical Analysis

- Mean: **353.01**
- Median: **0.37**
- Mode: **0.00**
- Standard Deviation: **2037.82**
- Skewness: **95.16**
- Kurtosis: **13,733.38**

### Observations

- The distribution is **extremely right-skewed**.
- The large difference between the mean (**353**) and median (**0.37**) indicates that a small number of extremely large values heavily influence the mean.
- Most customers have relatively low Debt Ratios, while only a few possess exceptionally large values.
- The feature is highly non-normal and may require transformation or clipping during preprocessing.

---

## Distribution Analysis

### Observations

- Most observations are concentrated below a Debt Ratio of **1.0**.
- A very long right tail is created by a small number of extremely large values.
- The distribution is heavily skewed and should not be summarized using the mean alone.
- Median and quantiles provide a much better representation of the typical customer.

---

## Outlier Analysis

- Default Rate (Non-Outliers): **6.96%**
- Default Rate (Outliers): **5.64%**

### Observations

- Unlike previous features, outliers do **not** correspond to a higher default rate.
- Customers with extremely large Debt Ratios actually exhibit a slightly lower default rate.
- These extreme values likely represent special financial situations (such as very low reported income) rather than increased credit risk.

**Decision:** Retain all outliers and investigate their nature during preprocessing.

---

## Relationship with Target

- Mean Debt Ratio (No Default): **357.15**
- Mean Debt Ratio (Default): **295.12**
- Median Debt Ratio (No Default): **0.363**
- Median Debt Ratio (Default): **0.428**

### Observations

- Median Debt Ratio is slightly higher among customers who default.
- The mean is misleading because it is dominated by extreme values.
- DebtRatio appears to have predictive value, although its relationship with default is weaker than the delinquency-related variables.

---

## Binning Analysis

| Debt Ratio | Default Rate |
|------------|-------------:|
| 0–0.10 | **5.95%** |
| 0.10–0.24 | **6.13%** |
| 0.24–0.37 | **5.48%** |
| 0.37–0.57 | **7.23%** |
| 0.57–76 | **9.57%** |
| >76 | **5.75%** |

### Observations

- Default risk generally increases as Debt Ratio rises from low to moderately high values.
- Customers with Debt Ratios between **0.57 and 76** have the highest observed default rate (**9.57%**).
- Surprisingly, extremely large Debt Ratios (>76) show a lower default rate.
- This indicates a **non-linear relationship** and suggests that extreme values may represent special financial circumstances rather than genuine increases in default risk.
- Business interpretation is therefore more informative than relying solely on correlation.

---

## Correlation Analysis

- Pearson Correlation: **-0.0076**
- Spearman Correlation: **0.0206**

### Observations

- Both correlation coefficients are very close to zero.
- This does **not** imply the feature is useless.
- The weak correlations arise because the relationship between DebtRatio and default is **non-linear** and heavily influenced by extreme values.
- Binning analysis demonstrates that the feature still contains meaningful predictive information.

---

## Feature Engineering Opportunities

Potential engineered features include:

- Log DebtRatio
- Winsorized / Clipped DebtRatio
- High Debt Burden Flag
- Debt Burden Categories
- Debt-to-Income Risk Bands
- Interaction with MonthlyIncome

---

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐⭐☆ High |
| Predictive Signal | ⭐⭐⭐☆☆ Moderate |
| Missing Values | None |
| Invalid Values | None |
| Distribution | Extremely Right-Skewed |
| Outliers | Retain & Investigate |
| Correlation | Very Weak (Non-linear Relationship) |
| Feature Engineering | Very High Priority |
| Preprocessing Recommendation | Investigate extreme values, consider clipping or transformation, and evaluate engineered debt burden features. |

### Final Verdict

`DebtRatio` is an important financial feature that measures a customer's debt burden. Although its raw correlation with default is weak, exploratory analysis reveals a meaningful **non-linear relationship** between debt burden and default risk. Extreme values heavily distort the distribution and should be carefully handled during preprocessing rather than removed. The feature offers strong opportunities for feature engineering and will be retained for model development.

In [ ]:
# ==========================================================
# Feature Audit: MonthlyIncome
# ==========================================================

feature_audit(
    df=df,
    feature="MonthlyIncome",
    target="SeriousDlqin2yrs"
)

# Feature Summary: MonthlyIncome

## Business Understanding

- `MonthlyIncome` represents the customer's total monthly income from all reported sources.
- Income reflects the customer's financial ability to repay debt and maintain regular loan payments.
- Customers with higher and more stable incomes generally have a lower probability of default.
- **Expected Relationship:** As monthly income increases, default risk is generally expected to decrease.

---

## Data Quality

- Missing Values: **29,731 (19.82%)**
- Negative Values: **0**
- Data Type: **float64**
- Unique Values: **13,594**
- Value Range: **0 – 3,008,750**

### Observations

- Nearly **one-fifth of the dataset contains missing income values**, making this one of the most important preprocessing challenges.
- No negative income values are present.
- Extremely high income values exist, creating a very wide range.
- Missing-value handling for this feature will require careful investigation during the preprocessing phase (MCAR, MAR, or MNAR).

---

## Statistical Analysis

- Mean: **6,670.22**
- Median: **5,400**
- Mode: **5,000**
- Standard Deviation: **14,384.67**
- Skewness: **114.04**
- Kurtosis: **19,503.89**

### Observations

- The distribution is **extremely right-skewed**.
- The mean is noticeably higher than the median because a small number of customers report exceptionally large incomes.
- Most customers earn between **3,400 and 8,249** per month, while only a few have very large incomes.
- The feature is highly non-normal and may benefit from log transformation or clipping during preprocessing.

---

## Distribution Analysis

### Observations

- Most customers are concentrated within relatively low-to-moderate income levels.
- A long right tail is produced by a small number of very high-income customers.
- Median income better represents the typical customer than the mean.
- The observed distribution is common for income-related variables.

---

## Outlier Analysis

- Default Rate (Non-Outliers): **6.75%**
- Default Rate (Outliers): **4.84%**

### Observations

- High-income outliers actually exhibit a **lower default rate** than the remaining population.
- These observations appear to represent genuinely wealthy customers rather than data errors.
- Removing these observations could eliminate useful predictive information.

**Decision:** Retain all outliers.

---

## Relationship with Target

- Mean Income (No Default): **6,747.84**
- Mean Income (Default): **5,630.83**
- Median Income (No Default): **5,466**
- Median Income (Default): **4,500**

### Observations

- Customers who default generally have lower monthly incomes.
- Both the mean and median consistently show lower income among defaulters.
- Income demonstrates meaningful predictive value and aligns with financial intuition.

---

## Binning Analysis

| Monthly Income | Default Rate |
|---------------|-------------:|
| ≤ 2,700 | **8.91%** |
| 2,700–4,000 | **9.04%** |
| 4,000–5,400 | **7.66%** |
| 5,400–7,080 | **6.39%** |
| 7,080–9,908 | **5.32%** |
| > 9,908 | **4.34%** |

### Observations

- Default risk steadily decreases as monthly income increases.
- Customers in the highest income group have approximately **half the default rate** of customers in the lowest income groups.
- The relationship is smooth and nearly monotonic, making MonthlyIncome an interpretable predictor of credit risk.
- Income categories may become valuable engineered features during later stages.

---

## Correlation Analysis

- Pearson Correlation: **-0.0197**
- Spearman Correlation: **-0.0670**

### Observations

- Both correlations are negative, indicating that higher income is associated with lower default risk.
- Spearman correlation is stronger than Pearson because the relationship is monotonic but affected by extreme income values.
- Although the numerical correlations are small, the binning analysis demonstrates a clear practical relationship between income and default.

---

## Feature Engineering Opportunities

Potential engineered features include:

- Log(MonthlyIncome)
- Income Categories
- High Income Flag
- Zero Income Flag
- Income per Dependent
- Income Stability Indicator (combined with other financial variables)

---

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐⭐⭐ Very High |
| Predictive Signal | ⭐⭐⭐⭐☆ High |
| Missing Values | **19.82% (Requires Investigation)** |
| Invalid Values | None |
| Distribution | Extremely Right-Skewed |
| Outliers | Informative (Keep) |
| Correlation | Weak Negative (Monotonic Relationship) |
| Feature Engineering | Very High Priority |
| Preprocessing Recommendation | Investigate missing-value mechanism, retain outliers, evaluate log transformation, and engineer income-based features. |

### Final Verdict

`MonthlyIncome` is a highly valuable financial feature that directly reflects a customer's repayment capacity. Despite its substantial missing-value rate and highly skewed distribution, exploratory analysis demonstrates a clear inverse relationship between income and default risk. Higher-income customers consistently exhibit lower default rates, while lower-income customers are considerably more likely to default. The feature should be retained, carefully imputed during preprocessing, and further enhanced through feature engineering.

In [ ]:
# ==========================================================
# Feature Audit: NumberOfOpenCreditLinesAndLoans
# ==========================================================

feature_audit(
    df=df,
    feature="NumberOfOpenCreditLinesAndLoans",
    target="SeriousDlqin2yrs"
)

# Feature Summary: NumberOfOpenCreditLinesAndLoans

## Business Understanding

- `NumberOfOpenCreditLinesAndLoans` represents the total number of active credit accounts held by a customer.
- These include credit cards, personal loans, auto loans, mortgages, and other active credit facilities.
- The feature reflects a customer's credit exposure and borrowing behavior.
- Having too few accounts may indicate limited credit history, while too many accounts may indicate excessive borrowing.
- **Expected Relationship:** Customers with a moderate number of open credit accounts are generally expected to have lower default risk, whereas very low or very high numbers may increase risk.

---

## Data Quality

- Missing Values: **0 (0.00%)**
- Negative Values: **0**
- Data Type: **int64**
- Unique Values: **58**
- Value Range: **0 – 58**

### Observations

- The feature contains no missing or invalid values.
- Values represent discrete counts of active credit accounts.
- The observed range appears reasonable for a credit-risk dataset.

---

## Statistical Analysis

- Mean: **8.45**
- Median: **8**
- Mode: **6**
- Standard Deviation: **5.15**
- Skewness: **1.22**
- Kurtosis: **3.09**

### Observations

- The distribution is **moderately right-skewed**.
- Mean and median are very close, indicating a relatively balanced distribution.
- Most customers have between **5 and 11** active credit accounts.
- Compared with previous financial variables, this feature is much less affected by extreme values.

---

## Distribution Analysis

### Observations

- Most customers maintain a moderate number of active credit accounts.
- Very high numbers of open accounts are relatively uncommon.
- The distribution appears suitable for modeling without major transformation.

---

## Outlier Analysis

- Default Rate (Non-Outliers): **6.68%**
- Default Rate (Outliers): **7.01%**

### Observations

- Customers with unusually high numbers of credit accounts exhibit only a slightly higher default rate.
- These observations appear to represent legitimate customer behavior rather than data errors.

**Decision:** Retain all outliers.

---

## Relationship with Target

- Mean Open Accounts (No Default): **8.49**
- Mean Open Accounts (Default): **7.88**
- Median Open Accounts (No Default): **8**
- Median Open Accounts (Default): **7**

### Observations

- Customers who default generally have slightly fewer active credit accounts.
- The difference between the two groups is relatively small, suggesting a moderate predictive signal.
- Credit history depth may contribute to default prediction when combined with other variables.

---

## Binning Analysis

| Open Credit Accounts | Default Rate |
|----------------------|-------------:|
| 0–4 | **9.22%** |
| 5–6 | **5.93%** |
| 7–8 | **5.24%** |
| 9–10 | **5.90%** |
| 11–13 | **5.82%** |
| >13 | **6.98%** |

### Observations

- Customers with **very few credit accounts (0–4)** have the highest default rate (**9.22%**).
- Default risk decreases as customers build a moderate credit history.
- Customers with approximately **7–8 active accounts** exhibit the lowest default rate.
- Risk increases slightly again among customers with a large number of open accounts (>13).
- The relationship follows a **U-shaped (non-linear)** pattern rather than a simple linear trend.

---

## Correlation Analysis

- Pearson Correlation: **-0.0297**
- Spearman Correlation: **-0.0386**

### Observations

- Both correlation coefficients are weakly negative.
- The weak correlations do not fully capture the feature's **U-shaped relationship** with default.
- Binning analysis provides a more meaningful interpretation than correlation alone.

---

## Feature Engineering Opportunities

Potential engineered features include:

- Low Credit History Flag
- High Credit Exposure Flag
- Credit Account Categories
- Sparse Credit Profile Indicator
- Interaction with DebtRatio
- Interaction with MonthlyIncome

---

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐⭐☆ High |
| Predictive Signal | ⭐⭐⭐☆☆ Moderate |
| Missing Values | None |
| Invalid Values | None |
| Distribution | Moderately Right-Skewed |
| Outliers | Informative (Keep) |
| Correlation | Weak Negative (U-Shaped Relationship) |
| Feature Engineering | High Priority |
| Preprocessing Recommendation | No missing-value treatment required. Consider engineering credit-account categories or low/high credit exposure indicators. |

### Final Verdict

`NumberOfOpenCreditLinesAndLoans` is a useful behavioral feature that reflects the breadth of a customer's credit history. EDA indicates a **U-shaped relationship** with default risk, where customers with very few active credit accounts exhibit the highest default rates, those with a moderate number of accounts show the lowest risk, and customers with many accounts experience a slight increase in risk. Although its linear correlation with the target is weak, the feature provides meaningful information and should be retained for model development and future feature engineering.

In [ ]:
# ==========================================================
# Feature Audit: NumberOfTimes90DaysLate
# ==========================================================

feature_audit(
    df=df,
    feature="NumberOfTimes90DaysLate",
    target="SeriousDlqin2yrs"
)

In [ ]:
# ==========================================================
# Custom Binning: NumberOfTimes90DaysLate
# ==========================================================

bins = [-1, 0, 1, 2, 5, 10, np.inf]
labels = ["0", "1", "2", "3-5", "6-10", ">10"]

df["Late90_Bin"] = pd.cut(
    df["NumberOfTimes90DaysLate"],
    bins=bins,
    labels=labels
)

bin_summary = (
    df.groupby("Late90_Bin", observed=True)["SeriousDlqin2yrs"]
      .agg(Customers="count", Defaults="sum")
)

bin_summary["Default_Rate"] = (
    bin_summary["Defaults"] / bin_summary["Customers"] * 100
).round(2)

display(bin_summary)

# ==========================================================
# Default Rate by 90-Day Delinquency Count
# ==========================================================

plt.figure(figsize=(7, 4))

sns.barplot(
    x=bin_summary.index,
    y=bin_summary["Default_Rate"]
)

plt.title("Default Rate by 90-Day Delinquency Count")
plt.xlabel("90+ Days Late Count")
plt.ylabel("Default Rate (%)")

plt.show()

# Feature Summary: NumberOfTimes90DaysLate

## Business Understanding

- `NumberOfTimes90DaysLate` represents the number of times a customer has been **90 or more days past due** on a credit payment.
- A 90-day delinquency is considered a severe repayment failure and is one of the strongest indicators of financial distress.
- Customers with repeated 90-day late payments are significantly more likely to default on future credit obligations.
- **Expected Relationship:** As the number of 90-day late payments increases, the probability of default is expected to increase substantially.

---

## Data Quality

- Missing Values: **0 (0.00%)**
- Negative Values: **0**
- Data Type: **int64**
- Unique Values: **19**
- Value Range: **0 – 98**

### Observations

- The feature contains no missing or invalid values.
- It is a discrete count variable representing severe delinquency events.
- Most customers have never experienced a 90-day delinquency, resulting in a highly imbalanced distribution.
- Very large values (e.g., 96 and 98 if confirmed) should be investigated during preprocessing, as they may represent coded values rather than actual delinquency counts.

---

## Statistical Analysis

- Mean: **0.266**
- Median: **0**
- Mode: **0**
- Standard Deviation: **4.17**
- Skewness: **23.09**
- Kurtosis: **537.72**

### Observations

- The distribution is extremely right-skewed.
- More than 75% of customers have never been 90 days late.
- A small number of customers have repeated severe delinquencies, producing a long right tail.
- The feature is highly non-normal, which is expected for delinquency count variables.

---

## Distribution Analysis

### Observations

- The majority of observations are concentrated at zero.
- The frequency decreases rapidly as the number of severe delinquencies increases.
- Customers with multiple 90-day delinquencies form a small but important high-risk group.

---

## Outlier Analysis

- Default Rate (Non-Outliers): **4.63%**
- Default Rate (Outliers): **41.64%**

### Observations

- Customers identified as outliers have an exceptionally high default rate.
- These outliers represent genuine high-risk borrowers rather than data errors.
- Removing them would significantly reduce the model's predictive power.

**Decision:** Retain all outliers.

---

## Relationship with Target

- Mean (No Default): **0.135**
- Mean (Default): **2.091**
- Median (No Default): **0**
- Median (Default): **0**

### Observations

- Customers who default have substantially more 90-day late payments on average.
- Although both groups have a median of zero, the large difference in means highlights the importance of repeated severe delinquencies.
- This feature provides a very strong signal for identifying high-risk customers.

---

## Binning Analysis

| 90-Day Late Payments | Default Rate |
|----------------------|-------------:|
| 0 | **4.63%** |
| 1 | **33.66%** |
| 2 | **49.90%** |
| 3–5 | **60.88%** |
| 6–10 | **68.07%** |
| >10 | **54.39%** |

### Observations

- Default risk increases dramatically after the first 90-day delinquency.
- Customers with a single severe delinquency already exhibit a default rate over **33%**, compared with only **4.63%** for customers with no severe delinquency.
- Risk continues to rise through the **6–10** late-payment category, reaching **68.07%**.
- The slight decline in the **>10** category is likely caused by the relatively small sample size and possible coded values (such as 96 and 98), rather than a genuine reduction in default risk.
- Overall, this feature demonstrates one of the strongest relationships with the target variable in the dataset.

---

## Correlation Analysis

- Pearson Correlation: **0.1172**
- Spearman Correlation: **0.3423**

### Observations

- Pearson correlation indicates a moderate positive linear relationship.
- Spearman correlation is considerably stronger, reflecting a strong monotonic increase in default risk as severe delinquencies accumulate.
- Among the features analyzed so far, this is one of the strongest correlations with the target.

---

## Feature Engineering Opportunities

Potential engineered features include:

- Any 90-Day Late Flag
- Severe Delinquency Flag
- Frequent 90-Day Delinquency Flag
- Total Late Payments (combined with 30–59 and 60–89 day delinquency features)
- Maximum Delinquency Indicator
- Delinquency Severity Categories

---

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐⭐⭐ Very High |
| Predictive Signal | ⭐⭐⭐⭐⭐ Very Strong |
| Missing Values | None |
| Invalid Values | None |
| Distribution | Extremely Right-Skewed |
| Outliers | Highly Informative (Keep) |
| Correlation | Strong Positive (Spearman = **0.3423**) |
| Feature Engineering | Very High Priority |
| Preprocessing Recommendation | Retain all observations, investigate extreme coded values (if present), and engineer delinquency-based risk indicators. |

### Final Verdict

`NumberOfTimes90DaysLate` is one of the most informative features in the dataset. Severe delinquency history has a clear and substantial impact on future credit default, with default rates increasing sharply as the number of 90-day late payments grows. The feature exhibits an extremely strong monotonic relationship with the target and should remain a core predictor throughout model development. Its outliers represent genuine high-risk customers rather than errors and should be preserved. During feature engineering, this variable should be used to create delinquency severity indicators and combined with the other delinquency variables to capture a customer's overall repayment behavior.

In [ ]:
# ==========================================================
# Feature Audit: NumberRealEstateLoansOrLines
# ==========================================================

feature_audit(
    df=df,
    feature="NumberRealEstateLoansOrLines",
    target="SeriousDlqin2yrs"
)

# Feature Summary: NumberRealEstateLoansOrLines

## Business Understanding

- `NumberRealEstateLoansOrLines` represents the total number of mortgages and real estate loans currently held by a customer.
- It reflects the customer's long-term borrowing commitments and ownership of financed real estate.
- Customers with one or two real estate loans often have established credit histories and stable financial profiles, whereas having no real estate loans or an unusually high number may indicate different risk characteristics.
- **Expected Relationship:** Customers with a moderate number of real estate loans are generally expected to have lower default risk, while having none or many loans may increase risk.

---

## Data Quality

- Missing Values: **0 (0.00%)**
- Negative Values: **0**
- Data Type: **int64**
- Unique Values: **28**
- Value Range: **0 – 54**

### Observations

- The feature contains no missing or invalid values.
- It is a discrete count variable.
- Most customers have between **0 and 2** real estate loans.
- A small number of customers hold a large number of real estate loans.

---

## Statistical Analysis

- Mean: **1.02**
- Median: **1**
- Mode: **0**
- Standard Deviation: **1.13**
- Skewness: **3.48**
- Kurtosis: **60.47**

### Observations

- The distribution is moderately to strongly right-skewed.
- Most observations are concentrated at small integer values.
- High kurtosis indicates a sharp peak with relatively few extreme observations.
- Compared with previous financial variables, skewness is much less severe.

---

## Distribution Analysis

### Observations

- The distribution contains **multiple peaks**, which is expected because this is a discrete count variable.
- Most customers have **0, 1, or 2** real estate loans, creating separate peaks instead of a smooth distribution.
- Therefore, the histogram is naturally multimodal rather than bell-shaped.
- This behavior is normal and does not indicate a data quality issue.

---

## Outlier Analysis

- Default Rate (Non-Outliers): **6.62%**
- Default Rate (Outliers): **17.91%**

### Observations

- Customers with unusually large numbers of real estate loans exhibit a much higher default rate.
- These observations likely represent customers with unusually high leverage rather than data errors.

**Decision:** Retain all outliers.

---

## Relationship with Target

- Mean (No Default): **1.020**
- Mean (Default): **0.989**
- Median (No Default): **1**
- Median (Default): **1**

### Observations

- The overall averages between the two groups are very similar.
- Mean and median alone suggest little predictive value.
- However, group averages hide the underlying non-linear relationship revealed through binning analysis.

---

## Binning Analysis

| Real Estate Loans | Default Rate |
|-------------------|-------------:|
| 0–1 | **6.84%** |
| 2 | **5.60%** |
| >2 | **8.45%** |

### Observations

- Customers with **two** real estate loans exhibit the lowest default rate.
- Customers with **no or only one** real estate loan have a slightly higher default rate.
- Customers with **more than two** real estate loans show the highest default rate.
- The feature demonstrates a **U-shaped relationship**, indicating that moderate real estate ownership is associated with lower default risk.

---

## Correlation Analysis

- Pearson Correlation: **-0.0070**
- Spearman Correlation: **-0.0341**

### Observations

- Both correlations are extremely weak.
- This does **not** imply the feature is useless.
- The weak correlations occur because the relationship with default is **non-linear**, making simple linear correlation an inadequate measure of predictive value.

---

## Feature Engineering Opportunities

Potential engineered features include:

- Homeowner Flag (Has Real Estate Loan)
- Multiple Property Owner Flag
- High Real Estate Exposure Flag
- Real Estate Loan Categories
- Interaction with MonthlyIncome
- Interaction with DebtRatio

---

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐☆☆ Moderate |
| Predictive Signal | ⭐⭐⭐☆☆ Moderate |
| Missing Values | None |
| Invalid Values | None |
| Distribution | Right-Skewed (Discrete Multi-Peak) |
| Outliers | Informative (Keep) |
| Correlation | Very Weak (U-Shaped Relationship) |
| Feature Engineering | Moderate Priority |
| Preprocessing Recommendation | No preprocessing required. Consider engineering homeowner and property-exposure indicators. |

### Final Verdict

`NumberRealEstateLoansOrLines` is a useful behavioral feature that captures a customer's real estate borrowing profile. Although its linear correlation with default is very weak, EDA reveals a meaningful **U-shaped relationship**, where customers with approximately two real estate loans exhibit the lowest default risk, while customers with very few or many loans show higher risk. The feature should be retained and explored further through feature engineering rather than removed based solely on its correlation.

In [ ]:
# ==========================================================
# Feature Audit: NumberOfTime60-89DaysPastDueNotWorse
# ==========================================================

feature_audit(
    df=df,
    feature="NumberOfTime60-89DaysPastDueNotWorse",
    target="SeriousDlqin2yrs"
)

In [ ]:
# ==========================================================
# Custom Binning: NumberOfTime60-89DaysPastDueNotWorse
# ==========================================================

bins = [-1, 0, 1, 2, 5, 10, np.inf]
labels = ["0", "1", "2", "3-5", "6-10", ">10"]

df["Late60_89_Bin"] = pd.cut(
    df["NumberOfTime60-89DaysPastDueNotWorse"],
    bins=bins,
    labels=labels
)

bin_summary = (
    df.groupby("Late60_89_Bin", observed=True)["SeriousDlqin2yrs"]
      .agg(
          Customers="count",
          Defaults="sum"
      )
)

bin_summary["Default_Rate"] = (
    bin_summary["Defaults"] / bin_summary["Customers"] * 100
).round(2)

display(bin_summary)

# ==========================================================
# Default Rate by 60–89 Day Delinquency Count
# ==========================================================

plt.figure(figsize=(7,4))

sns.barplot(
    x=bin_summary.index,
    y=bin_summary["Default_Rate"]
)

plt.title("Default Rate by 60–89 Day Delinquency Count")
plt.xlabel("60–89 Days Late Count")
plt.ylabel("Default Rate (%)")

plt.tight_layout()
plt.show()

# Feature Summary: NumberOfTime60-89DaysPastDueNotWorse

## Business Understanding

- `NumberOfTime60-89DaysPastDueNotWorse` represents the number of times a customer has been **60–89 days past due** on a credit payment.
- A 60–89 day delinquency indicates significant repayment difficulty and is a strong warning sign before a severe default event.
- Customers with repeated 60–89 day late payments are considerably more likely to default on future credit obligations.
- **Expected Relationship:** As the number of 60–89 day late payments increases, the probability of default is expected to increase substantially.

---

## Data Quality

- Missing Values: **0 (0.00%)**
- Negative Values: **0**
- Data Type: **int64**
- Unique Values: **13**
- Value Range: **0 – 98**

### Observations

- No missing or invalid values are present.
- The feature is a discrete count variable.
- Most customers have never experienced a 60–89 day delinquency.
- Extremely large values (e.g., 96 and 98, if confirmed) should be investigated during preprocessing because they may represent coded values rather than actual delinquency counts.

---

## Statistical Analysis

- Mean: **0.240**
- Median: **0**
- Mode: **0**
- Standard Deviation: **4.16**
- Skewness: **23.33**
- Kurtosis: **545.66**

### Observations

- The distribution is extremely right-skewed.
- More than 75% of customers have zero 60–89 day late payments.
- A small number of customers account for repeated delinquencies, producing a long right tail.
- This highly skewed distribution is expected for delinquency count variables.

---

## Distribution Analysis

### Observations

- The majority of observations are concentrated at zero.
- The frequency decreases rapidly as the delinquency count increases.
- Customers with repeated 60–89 day delinquencies form a relatively small but high-risk group.

---

## Outlier Analysis

- Default Rate (Non-Outliers): **5.10%**
- Default Rate (Outliers): **36.43%**

### Observations

- Customers classified as outliers have a dramatically higher default rate.
- These observations represent genuine high-risk borrowers rather than data errors.
- Removing them would reduce valuable predictive information.

**Decision:** Retain all outliers.

---

## Relationship with Target

- Mean (No Default): **0.127**
- Mean (Default): **1.828**
- Median (No Default): **0**
- Median (Default): **0**

### Observations

- Customers who default have substantially more 60–89 day delinquencies on average.
- Although the median is zero for both groups, the large difference in means demonstrates the importance of repeated late payments.
- This feature provides a strong predictive signal for default risk.

---

## Binning Analysis

| 60–89 Day Late Payments | Default Rate |
|-------------------------|-------------:|
| 0 | **5.10%** |
| 1 | **31.01%** |
| 2 | **50.18%** |
| 3–5 | **58.21%** |
| 6–10 | **64.29%** |
| >10 | **54.81%** |

### Observations

- Default risk increases sharply after the first 60–89 day delinquency.
- Customers with one delinquency already exhibit a default rate above **31%**, compared with only **5.10%** for customers with no delinquency.
- Risk continues increasing through the **6–10** category, reaching **64.29%**.
- The slight decline in the **>10** category is likely due to the very small sample size and possible coded values (such as 96 and 98) rather than a genuine reduction in risk.
- Overall, this feature demonstrates a strong monotonic relationship with default.

---

## Correlation Analysis

- Pearson Correlation: **0.1023**
- Spearman Correlation: **0.2771**

### Observations

- Pearson correlation indicates a moderate positive linear relationship.
- Spearman correlation is substantially stronger, showing that default risk generally increases as the number of 60–89 day delinquencies increases.
- This feature is among the strongest predictors analyzed during EDA.

---

## Feature Engineering Opportunities

Potential engineered features include:

- Any 60–89 Day Late Flag
- Repeated Delinquency Flag
- Chronic Delinquency Indicator
- Total Late Payments (combined with 30–59 and 90+ day delinquency variables)
- Delinquency Severity Categories
- Overall Credit Behavior Score

---

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐⭐⭐ Very High |
| Predictive Signal | ⭐⭐⭐⭐⭐ Very Strong |
| Missing Values | None |
| Invalid Values | None |
| Distribution | Extremely Right-Skewed |
| Outliers | Highly Informative (Keep) |
| Correlation | Strong Positive (Spearman = **0.2771**) |
| Feature Engineering | Very High Priority |
| Preprocessing Recommendation | Retain all observations, investigate unusually large coded values, and engineer delinquency-based risk indicators. |

### Final Verdict

`NumberOfTime60-89DaysPastDueNotWorse` is one of the strongest predictors of credit default in the dataset. Customers with repeated 60–89 day delinquencies experience a substantial increase in default risk, making this feature highly valuable for predictive modeling. Its extreme skewness and concentration of zero values are expected characteristics of delinquency data rather than quality issues. The feature should be retained, its informative outliers preserved, and it should play a key role in future feature engineering alongside the other delinquency variables.

In [ ]:
# ==========================================================
# Feature Audit: NumberOfDependents
# ==========================================================

feature_audit(
    df=df,
    feature="NumberOfDependents",
    target="SeriousDlqin2yrs"
)

# Feature Summary: NumberOfDependents

## Business Understanding

- `NumberOfDependents` represents the number of individuals who financially depend on the customer, such as children, spouse, or other family members.
- A larger number of dependents generally increases household expenses and financial responsibilities.
- Customers supporting more dependents may experience greater financial pressure, reducing their ability to meet debt obligations.
- **Expected Relationship:** As the number of dependents increases, default risk is expected to increase gradually.

---

## Data Quality

- Missing Values: **3,924 (2.62%)**
- Negative Values: **0**
- Data Type: **float64**
- Unique Values: **13**
- Value Range: **0 – 20**

### Observations

- This is the only feature (besides `MonthlyIncome`) containing missing values.
- Missingness is relatively low (2.62%) and can be handled during preprocessing.
- The feature is stored as `float64` because of missing values, although it represents discrete count data.
- No invalid or negative values were found.

---

## Statistical Analysis

- Mean: **0.757**
- Median: **0**
- Mode: **0**
- Standard Deviation: **1.115**
- Skewness: **1.59**
- Kurtosis: **3.00**

### Observations

- The distribution is moderately right-skewed.
- Most customers have no dependents.
- Extreme values exist but are relatively rare.
- Compared with previous financial variables, the distribution is much closer to normal.

---

## Distribution Analysis

### Observations

- The distribution consists of discrete integer counts, producing multiple peaks in the histogram.
- Most customers have **0 or 1 dependent**.
- The frequency decreases steadily as the number of dependents increases.
- This behavior is expected for household demographic data and does not indicate data quality issues.

---

## Outlier Analysis

- Default Rate (Non-Outliers): **6.43%**
- Default Rate (Outliers): **9.25%**

### Observations

- Customers with unusually large numbers of dependents exhibit a somewhat higher default rate.
- The increase is noticeable but much smaller than that observed for delinquency-related features.
- These observations represent valid customer profiles and should be retained.

**Decision:** Retain all outliers.

---

## Relationship with Target

- Mean (No Default): **0.743**
- Mean (Default): **0.948**
- Median (No Default): **0**
- Median (Default): **0**

### Observations

- Customers who default have slightly more dependents on average.
- The difference between the two groups is relatively small.
- This suggests that the feature contributes useful information but is not a dominant predictor of default.

---

## Binning Analysis

| Number of Dependents | Default Rate |
|----------------------|-------------:|
| 0–1 | **6.21%** |
| 2 | **8.11%** |
| >2 | **9.25%** |

### Observations

- Default risk increases gradually as the number of dependents increases.
- Customers with **0–1 dependents** have the lowest default rate.
- Customers with **more than two dependents** exhibit the highest default rate.
- Unlike the delinquency variables, the increase is modest but consistent, indicating a positive relationship with default.

---

## Correlation Analysis

- Pearson Correlation: **0.0460**
- Spearman Correlation: **0.0460**

### Observations

- Both correlations are weak but positive.
- The feature has limited standalone predictive power.
- Despite the weak correlation, the increasing default rate across dependency groups suggests it may still provide useful complementary information when combined with other financial features.

---

## Feature Engineering Opportunities

Potential engineered features include:

- Has Dependents Flag
- Large Family Indicator
- High Dependency Burden Flag
- Income per Dependent (combined with `MonthlyIncome`)
- Debt per Dependent Ratio
- Family Financial Burden Score

---

## Final Decision

| Criterion | Decision |
|------------|----------|
| Keep Feature | ✅ Yes |
| Remove Feature | ❌ No |
| Business Importance | ⭐⭐⭐☆☆ Moderate |
| Predictive Signal | ⭐⭐☆☆☆ Weak to Moderate |
| Missing Values | 2.62% (Requires Imputation) |
| Invalid Values | None |
| Distribution | Moderately Right-Skewed |
| Outliers | Informative (Keep) |
| Correlation | Weak Positive |
| Feature Engineering | Moderate Priority |
| Preprocessing Recommendation | Impute missing values, retain all observations, and consider creating dependency-based financial burden features. |

### Final Verdict

`NumberOfDependents` provides useful demographic information about a customer's financial responsibilities. Although its individual predictive power is relatively weak compared with delinquency and credit behavior variables, EDA shows a consistent increase in default risk as the number of dependents grows. The feature should be retained, its missing values imputed during preprocessing, and it should be considered for interaction features with `MonthlyIncome`, `DebtRatio`, and other financial variables to better capture household financial burden.

# multivariate analysis


In [ ]:
# ==========================================================
# Pearson Correlation Heatmap
# ==========================================================

plt.figure(figsize=(11,8))

corr = df.corr(numeric_only=True, method="pearson")


sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Pearson Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Spearman Correlation Heatmap
# ==========================================================

plt.figure(figsize=(11,8))

corr = df.corr(numeric_only=True, method="spearman")

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Spearman Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Correlation with Target
# ==========================================================

pearson_target = (
    df.corr(numeric_only=True, method="pearson")
      ["SeriousDlqin2yrs"]
      .drop("SeriousDlqin2yrs")
      .sort_values(key=abs, ascending=False)
)

spearman_target = (
    df.corr(numeric_only=True, method="spearman")
      ["SeriousDlqin2yrs"]
      .drop("SeriousDlqin2yrs")
      .sort_values(key=abs, ascending=False)
)

comparison = pd.DataFrame({
    "Pearson": pearson_target,
    "Spearman": spearman_target
})

display(comparison)

In [ ]:
# ==========================================================
# Highly Correlated Feature Pairs
# ==========================================================

corr = df.corr(numeric_only=True).abs()

upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

high_corr = (
    upper.stack()
         .reset_index()
)

high_corr.columns = [
    "Feature 1",
    "Feature 2",
    "Correlation"
]

high_corr = (
    high_corr
    .query("Correlation >= 0.70")
    .sort_values("Correlation", ascending=False)
)

display(high_corr)

In [ ]:
!pip3 install statsmodels

In [ ]:
# ==========================================================
# Variance Inflation Factor (VIF)
# ==========================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df.drop(columns=["SeriousDlqin2yrs"]).select_dtypes(include=np.number).dropna()

vif = pd.DataFrame()

vif["Feature"] = X.columns

vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

vif = vif.sort_values("VIF", ascending=False)

display(vif)

# Bivariate Analaysis

In [ ]:
# ==========================================================
# Scatter Plot: MonthlyIncome vs DebtRatio
# ==========================================================

plt.figure(figsize=(7,5))

sns.scatterplot(
    data=df,
    x="MonthlyIncome",
    y="DebtRatio",
    alpha=0.3
)

plt.title("Monthly Income vs Debt Ratio")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Scatter Plot: Revolving Utilization vs Debt Ratio
# ==========================================================

plt.figure(figsize=(7,5))

sns.scatterplot(
    data=df,
    x="RevolvingUtilizationOfUnsecuredLines",
    y="DebtRatio",
    alpha=0.3
)

plt.title("Revolving Utilization vs Debt Ratio")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Scatter Plot: Credit Lines vs Real Estate Loans
# ==========================================================

plt.figure(figsize=(7,5))

sns.scatterplot(
    data=df,
    x="NumberOfOpenCreditLinesAndLoans",
    y="NumberRealEstateLoansOrLines",
    alpha=0.3
)

plt.title("Credit Lines vs Real Estate Loans")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Pairplot of Important Features
# ==========================================================

cols = [
    "MonthlyIncome",
    "DebtRatio",
    "age",
    "RevolvingUtilizationOfUnsecuredLines",
    "SeriousDlqin2yrs"
]

sns.pairplot(
    df[cols],
    hue="SeriousDlqin2yrs",
    corner=True
)

plt.show()

In [ ]:
# ==========================================================
# Crosstab: 30-59 vs 90 Days Late
# ==========================================================

pd.crosstab(
    df["NumberOfTime30-59DaysPastDueNotWorse"],
    df["NumberOfTimes90DaysLate"]
)

# Step 1.5: Bivariate Analysis Summary

## 1. Monthly Income vs Debt Ratio

- The scatter plot is dominated by extreme outliers in both variables, causing most observations to be compressed into the lower-left corner.
- No clear positive or negative relationship is visible between Monthly Income and Debt Ratio.
- Customers with high income may have either low or high debt ratios, while customers with low income also exhibit a wide range of debt ratios.
- The plot suggests these variables measure different aspects of a customer's financial profile.
- A zoomed-in (0–99th percentile) or log-transformed visualization would provide a clearer view of the majority of customers.

---

## 2. Revolving Utilization vs Debt Ratio

- Similar to the previous plot, extreme outliers dominate both axes and compress the majority of observations.
- No obvious linear or nonlinear relationship is visible in the full-scale plot.
- Customers with high revolving credit utilization do not necessarily have high overall debt ratios.
- Revolving Utilization and Debt Ratio capture different dimensions of financial behavior and therefore provide complementary information.
- Visualization after removing extreme values or applying log scaling would improve interpretability.

---

## 3. Open Credit Lines vs Real Estate Loans

- Most customers have between **10–25 open credit lines** and **0–5 real estate loans**.
- A mild positive relationship exists for customers with fewer than approximately **25 credit lines**, where an increase in total credit accounts is often accompanied by a slight increase in real estate loans.
- Beyond this range, the variability increases considerably, indicating that customers with many credit accounts may still have very different numbers of real estate loans.
- The triangular distribution occurs because the number of real estate loans cannot exceed the total number of credit accounts.
- A few extreme observations (customers with 50+ credit lines and many real estate loans) are rare but valid outliers.
- Overall, these two variables are not strongly correlated and describe different aspects of customer borrowing behavior.

---

## 4. Pairplot Analysis

- The diagonal KDE plots show the individual distribution of each feature for default and non-default customers.
- Younger customers exhibit a slightly higher probability of default, as observed from the age distribution.
- Most feature pairs show substantial overlap between default and non-default customers, indicating that no single pair of variables can perfectly separate the two classes.
- No obvious clusters or naturally separable customer groups are observed.
- Scatter plots involving highly skewed variables are heavily influenced by extreme outliers, reducing their interpretability.
- The pairplot confirms that the dataset contains complex relationships that require machine learning models rather than simple threshold-based rules.

---

## 5. Delinquency Crosstab Analysis

- Customers with more **30–59 day late payments** generally also have more **90+ day late payments**.
- The highest frequencies occur near the diagonal of the contingency table, indicating that the delinquency variables increase together.
- This strongly supports the extremely high Pearson correlations (>0.98) observed between the three delinquency features.
- Values such as **96** and **98** appear as isolated categories and are likely coded values rather than actual delinquency counts.
- The three delinquency variables contain substantial overlapping information and should be evaluated carefully for multicollinearity during modeling.

---

## Overall Bivariate Analysis Conclusion

- Most non-delinquency financial variables exhibit weak pairwise relationships.
- Delinquency-related features are the only variables showing extremely strong associations with one another.
- Heavy skewness and extreme outliers reduce the usefulness of several scatter plots at full scale.
- Most financial variables capture different aspects of customer behavior rather than redundant information.
- Feature interactions may still improve predictive performance even when pairwise correlations are weak.

In [ ]:
# ==========================================================
# Joint Plot
# ==========================================================

sns.jointplot(
    data=df,
    x="MonthlyIncome",
    y="DebtRatio",
    kind="scatter",
    height=6
)

plt.show()

In [ ]:
# ==========================================================
# Monthly Income vs Debt Ratio (0–99th Percentile)
# ==========================================================

plot_df = df[
    (df["MonthlyIncome"] <= df["MonthlyIncome"].quantile(0.99)) &
    (df["DebtRatio"] <= df["DebtRatio"].quantile(0.99))
]

plt.figure(figsize=(7,5))

sns.scatterplot(
    data=plot_df,
    x="MonthlyIncome",
    y="DebtRatio",
    alpha=0.35,
    s=25
)

plt.title("Monthly Income vs Debt Ratio (0–99th Percentile)")
plt.xlabel("Monthly Income")
plt.ylabel("Debt Ratio")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Revolving Utilization vs Debt Ratio (0–99th Percentile)
# ==========================================================

plot_df = df[
    (df["RevolvingUtilizationOfUnsecuredLines"]
        <= df["RevolvingUtilizationOfUnsecuredLines"].quantile(0.99)) &
    (df["DebtRatio"]
        <= df["DebtRatio"].quantile(0.99))
]

plt.figure(figsize=(7,5))

sns.scatterplot(
    data=plot_df,
    x="RevolvingUtilizationOfUnsecuredLines",
    y="DebtRatio",
    alpha=0.35,
    s=25
)

plt.title("Revolving Utilization vs Debt Ratio (0–99th Percentile)")
plt.xlabel("Revolving Utilization")
plt.ylabel("Debt Ratio")

plt.tight_layout()
plt.show()

In [ ]:
plot_df = df[
    (df["MonthlyIncome"] <= df["MonthlyIncome"].quantile(0.99)) &
    (df["DebtRatio"] <= df["DebtRatio"].quantile(0.99))
]

plt.figure(figsize=(7,5))

plt.hexbin(
    plot_df["MonthlyIncome"],
    plot_df["DebtRatio"],
    gridsize=40,
    cmap="Blues",
    mincnt=1
)

plt.colorbar(label="Customers")

plt.title("Monthly Income vs Debt Ratio (99th Percentile)")
plt.xlabel("Monthly Income")
plt.ylabel("Debt Ratio")

plt.show()

In [ ]:
plot_df = df[
    (df["RevolvingUtilizationOfUnsecuredLines"] <= df["RevolvingUtilizationOfUnsecuredLines"].quantile(0.99)) &
    (df["DebtRatio"] <= df["DebtRatio"].quantile(0.99))
]

plt.figure(figsize=(7,5))

plt.hexbin(
    plot_df["RevolvingUtilizationOfUnsecuredLines"],
    plot_df["DebtRatio"],
    gridsize=40,
    cmap="Blues",
    mincnt=1
)

plt.colorbar(label="Customers")

plt.title("Revolving Utilization vs Debt Ratio (99th Percentile)")
plt.xlabel("Revolving Utilization")
plt.ylabel("Debt Ratio")

plt.show()

# Step 1.5: Bivariate Analysis

## Objective

The purpose of bivariate analysis is to examine relationships between pairs of variables and identify patterns that are not visible during univariate analysis.

This step helps answer the following questions:

- Are two features strongly related?
- Do some features contain redundant information?
- Which variables exhibit multicollinearity?
- Are there interaction patterns that may improve predictive models?
- Do the visual relationships agree with statistical correlation measures?

---

# 1. Monthly Income vs Debt Ratio

### Scatter Plot (Original)

- The original scatter plot is dominated by extreme outliers, causing almost all observations to be compressed into the lower-left corner.
- Due to severe overplotting, no meaningful relationship can be interpreted from the raw visualization.

### Hexbin Plot (0–99th Percentile)

- The hexbin plot clearly reveals the density of the majority of customers.
- Most customers earn approximately **$3,000–$8,000 per month**.
- The highest customer density occurs at **Debt Ratios below 1**.
- No clear positive or negative relationship exists between Monthly Income and Debt Ratio.
- Customers with both low and high incomes may exhibit either low or high Debt Ratios.
- Extremely high Debt Ratios primarily occur among customers with very low incomes, likely because Debt Ratio uses income in the denominator.
- The visualization is consistent with the near-zero Pearson and weak Spearman correlations, confirming that Monthly Income alone is not a strong determinant of Debt Ratio.

---

# 2. Revolving Utilization vs Debt Ratio

### Scatter Plot (Original)

- The original scatter plot suffers from severe overplotting because of the large dataset.
- The point cloud forms a nearly rectangular distribution without any visible linear or nonlinear trend.
- Dense vertical bands appear near **0% utilization** and **100% utilization**, indicating many customers fall into these utilization levels.
- Customers with identical Revolving Utilization values may have vastly different Debt Ratios.

### Hexbin Plot (0–99th Percentile)

- The highest customer density occurs at **low Revolving Utilization (0–10%)** and **Debt Ratios below 1**.
- No consistent increase or decrease in Debt Ratio is observed as Revolving Utilization changes.
- High Debt Ratios occur across nearly the entire utilization range.
- Revolving Utilization measures credit card usage, whereas Debt Ratio measures total debt burden; therefore, the two variables capture different aspects of customer financial behavior.
- The visualization supports the near-zero Pearson and weak Spearman correlations.

---

# 3. Open Credit Lines vs Real Estate Loans

- Most customers possess between **10–25 open credit accounts** and **0–5 real estate loans**.
- A mild positive relationship exists among customers with fewer than approximately **25 credit accounts**, where additional credit lines are generally accompanied by slightly more real estate loans.
- Beyond this range, variability increases substantially, indicating that customers with many credit accounts may still have very different numbers of real estate loans.
- The triangular distribution occurs because the number of real estate loans cannot exceed the total number of open credit lines.
- A few customers with exceptionally large numbers of credit accounts represent rare but valid observations.
- Overall, Pearson and Spearman correlations remain weak, indicating that these features provide complementary rather than redundant information.

---

# 4. Pairplot Analysis

- The diagonal KDE plots illustrate the distribution of each feature separately for defaulted and non-defaulted customers.
- Younger customers exhibit slightly higher default rates than older customers.
- Most feature pairs show considerable overlap between default and non-default classes, indicating that no single feature pair can perfectly separate the target classes.
- Highly skewed variables remain dominated by extreme observations even after visualization.
- The pairplot suggests that predicting default requires combining multiple variables rather than relying on simple thresholds.

---

# 5. Delinquency Crosstab Analysis

The contingency table comparing delinquency features reveals a very strong association among the three late-payment variables.

Observations:

- Customers with many **30–59 day late payments** usually also have many **60–89 day** and **90+ day** late payments.
- The largest counts lie along the diagonal of the contingency table, demonstrating that delinquency severity increases together.
- Encoded values such as **96** and **98** appear as isolated categories and are likely special codes rather than literal delinquency counts.
- These observations strongly support the extremely high pairwise correlations calculated later.

---

# 6. Correlation Analysis

### Correlation with Target

| Observation | Interpretation |
|-------------|---------------|
| Age | Moderate negative relationship with default. Younger customers default more frequently. |
| Revolving Utilization | Moderate monotonic positive relationship with default. |
| 30–59, 60–89, 90 Days Late | Strongest positive relationships with default. |
| Monthly Income | Weak negative relationship. |
| Debt Ratio | Nearly no relationship. |
| Credit Lines | Very weak relationship. |
| Real Estate Loans | Very weak relationship. |
| Dependents | Weak positive relationship. |

---

### Feature-to-Feature Correlation

The strongest correlations are:

- NumberOfTime30-59DaysPastDueNotWorse ↔ NumberOfTime60-89DaysPastDueNotWorse
- NumberOfTime30-59DaysPastDueNotWorse ↔ NumberOfTimes90DaysLate
- NumberOfTime60-89DaysPastDueNotWorse ↔ NumberOfTimes90DaysLate

All three exhibit correlations above **0.98**, indicating extremely strong redundancy.

Most remaining feature pairs show only weak relationships.

---

# 7. Multicollinearity (VIF)

Variance Inflation Factor (VIF) confirms the correlation analysis.

Observations:

- The three delinquency variables exhibit extremely high VIF values, indicating severe multicollinearity.
- Remaining variables possess acceptable VIF values and therefore do not suffer from significant multicollinearity.
- Tree-based algorithms (Random Forest, XGBoost, LightGBM, CatBoost) generally tolerate multicollinearity well.
- Linear models may require regularization or feature engineering to address highly correlated delinquency variables.

---

# Overall Conclusions

### Strong Predictors

- NumberOfTimes90DaysLate
- NumberOfTime60-89DaysPastDueNotWorse
- NumberOfTime30-59DaysPastDueNotWorse
- RevolvingUtilizationOfUnsecuredLines
- Age

---

### Weak Predictors

- DebtRatio
- MonthlyIncome
- NumberRealEstateLoansOrLines
- NumberOfOpenCreditLinesAndLoans
- NumberOfDependents

---

### Key Findings

- Delinquency history is the strongest indicator of future default.
- Revolving credit utilization contributes useful predictive information.
- Younger customers exhibit higher default rates.
- Most financial variables capture different aspects of customer behavior despite weak pairwise correlations.
- Severe multicollinearity exists only among the three delinquency variables.
- Heavy skewness and extreme outliers affect several financial features, requiring robust preprocessing and careful visualization.
- Tree-based machine learning models are well suited for this dataset because they naturally capture nonlinear relationships and interactions while remaining robust to multicollinearity and outliers.

# Step 1.5 — Multivariate Analysis

The objective of multivariate analysis is to understand how variables relate to one another, identify redundant information, detect multicollinearity, and discover opportunities for feature engineering.

---

# 1. Correlation with Target

| Feature | Pearson | Spearman |
|---------|---------:|---------:|
| NumberOfTimes90DaysLate | 0.117 | **0.342** |
| NumberOfTime60-89DaysPastDueNotWorse | 0.102 | **0.277** |
| NumberOfTime30-59DaysPastDueNotWorse | 0.126 | **0.257** |
| RevolvingUtilizationOfUnsecuredLines | -0.002 | **0.240** |
| age | -0.115 | -0.117 |
| is_outlier | 0.069 | 0.069 |
| MonthlyIncome | -0.020 | -0.067 |
| NumberOfDependents | 0.046 | 0.046 |
| NumberOfOpenCreditLinesAndLoans | -0.030 | -0.039 |
| NumberRealEstateLoansOrLines | -0.007 | -0.034 |
| DebtRatio | -0.008 | 0.021 |

## Observations

- The three delinquency variables exhibit the strongest relationships with the target.
- `NumberOfTimes90DaysLate` is the single most predictive feature based on Spearman correlation.
- `RevolvingUtilizationOfUnsecuredLines` has almost zero Pearson correlation but a strong Spearman correlation, indicating a non-linear monotonic relationship.
- Age has a moderate negative relationship, suggesting younger customers are more likely to default.
- The remaining demographic and financial variables individually contribute relatively weak predictive signals.

---

# 2. Highly Correlated Feature Pairs

| Feature 1 | Feature 2 | Correlation |
|------------|-----------|------------:|
| NumberOfTimes90DaysLate | NumberOfTime60-89DaysPastDueNotWorse | **0.993** |
| NumberOfTime30-59DaysPastDueNotWorse | NumberOfTime60-89DaysPastDueNotWorse | **0.987** |
| NumberOfTime30-59DaysPastDueNotWorse | NumberOfTimes90DaysLate | **0.984** |

## Observations

- The three delinquency variables are almost perfectly correlated.
- They capture nearly identical aspects of customer repayment behavior.
- This indicates substantial redundancy among these variables.
- Such high correlations strongly suggest multicollinearity.

---

# 3. Multicollinearity (VIF)

| Feature | VIF |
|---------|----:|
| NumberOfTime60-89DaysPastDueNotWorse | **60.55** |
| NumberOfTimes90DaysLate | **49.28** |
| NumberOfTime30-59DaysPastDueNotWorse | **27.02** |
| NumberOfOpenCreditLinesAndLoans | 4.77 |
| age | 3.93 |
| NumberRealEstateLoansOrLines | 2.30 |
| NumberOfDependents | 1.47 |
| MonthlyIncome | 1.24 |
| DebtRatio | 1.01 |
| RevolvingUtilizationOfUnsecuredLines | 1.00 |

## Observations

- All three delinquency variables have extremely high VIF values.
- Their information overlaps considerably, resulting in severe multicollinearity.
- The remaining variables exhibit acceptable VIF values and do not present multicollinearity concerns.

---

# 4. Redundant Information

The following variables contain overlapping information:

- NumberOfTime30-59DaysPastDueNotWorse
- NumberOfTime60-89DaysPastDueNotWorse
- NumberOfTimes90DaysLate

These variables all measure customer delinquency at different severity levels.

Although statistically redundant, each represents a different stage of repayment behavior and contains meaningful business information.

**Decision:** Retain all three features during EDA and evaluate feature importance during model development before considering removal.

---

# 5. Interaction Opportunities

Potential interaction features include:

### Credit Behavior

- Total Late Payments
- Total Severe Delinquencies
- Maximum Delinquency
- Delinquency Severity Score
- Any Late Payment Flag

### Financial Burden

- DebtRatio × MonthlyIncome
- MonthlyIncome ÷ NumberOfDependents
- DebtRatio per Dependent

### Credit Utilization

- RevolvingUtilization × DebtRatio
- RevolvingUtilization × Open Credit Lines

### Asset Profile

- Real Estate Loans × MonthlyIncome
- Real Estate Loans × DebtRatio

---

# 6. Final Conclusions

- Delinquency history is the strongest predictor of future default.
- The three delinquency variables exhibit severe multicollinearity but remain highly informative.
- Spearman correlation provides a much better representation of relationships than Pearson because many variables are highly skewed and non-linear.
- Most remaining features contribute complementary rather than redundant information.
- No additional feature pairs demonstrate problematic correlation.

---

# 7. Recommendations for Preprocessing

- Impute missing values in `MonthlyIncome` and `NumberOfDependents`.
- Preserve outliers because they represent genuine high-risk customers.
- Investigate unusually large delinquency values (e.g., 96 and 98) before model training.
- Consider engineering a combined delinquency score while also evaluating the original delinquency variables separately.
- Evaluate feature importance using tree-based models before removing correlated features.